# 01 — Environment Walkthrough

**Milestone 1 · Notebook 1**

Verifies the complete Milestone 1 setup:
- Environment resets and steps correctly
- All three baselines produce sensible numbers  
- Four visualisations: price, SoC, actions, cumulative profit
- Sanity check: `random < threshold < LP`

**GitHub:** https://github.com/MoFarahani2043610/multi_batteries_ppo_sac

In [ ]:
import sys, os

# Add project paths
for p in [
    os.path.abspath('..'),
    os.path.abspath(os.path.join('..', 'env')),
    os.path.abspath(os.path.join('..', 'baselines')),
    os.path.abspath(os.path.join('..', 'data')),
    os.path.abspath(os.path.join('..', 'visualisation')),
]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

try:
    import storage_arbitrage_env as _sae
    print('OK  storage_arbitrage_env found')
except ImportError as e:
    print('ERROR:', e)

print('Python:', sys.version[:6])
print('NumPy :', np.__version__)

## 1. Import the Environment

In [ ]:
from storage_arbitrage_env import (
    StorageArbitrageEnv, BatteryConfig,
    SyntheticPriceSource, HistoricalPriceSource
)

env = StorageArbitrageEnv(n_batteries=1)
print(env)
print(f'  Observation space : {env.observation_space}')
print(f'  Action space      : {env.action_space}')

## 2. Reset and Step — Basic Sanity Check

The episode length is T = (24 × 60) ÷ 5 = **288 timesteps** (one full day at 5-minute resolution).

In [ ]:
obs, info = env.reset(seed=42)
print(f'Initial obs   : {obs}')
print(f'Initial SoC   : {info["soc_mwh"]} MWh')
print(f'Initial price : ${info["prices"][0]:.2f}/MWh')

action = env.action_space.sample()
obs2, reward, terminated, truncated, info2 = env.step(action)
print(f'\nAction        : {action[0]:+.4f} MW  (+ charge, - discharge)')
print(f'Reward        : ${reward:.4f}')
print(f'New SoC       : {info2["soc_mwh"]} MWh')
print(f'Cash flow     : ${info2["cash_flow"]:.4f}')
print(f'Episode done  : {terminated or truncated}')

## 3. Scale from N=1 to N=20

Observation and action spaces grow automatically with N — no code changes needed.

In [ ]:
print(f'  {"N":>4}   obs shape     action shape')
print('  ' + '-'*38)
for N in [1, 2, 5, 10, 20]:
    e = StorageArbitrageEnv(n_batteries=N)
    obs, _ = e.reset(seed=0)
    print(f'  {N:>4}   {str(obs.shape):<14}  {str(e.action_space.shape)}')
    e.close()

## 4. Heterogeneous Battery Fleet

Each battery can have different capacity, power limits, and efficiency.

In [ ]:
fleet = [
    BatteryConfig(capacity_mwh=2.0, p_charge_max=1.0, p_discharge_max=1.0,
                  eta_charge=0.95, eta_discharge=0.95),   # large efficient battery
    BatteryConfig(capacity_mwh=0.5, p_charge_max=0.3, p_discharge_max=0.3,
                  eta_charge=0.85, eta_discharge=0.85),   # small older battery
    BatteryConfig(capacity_mwh=1.0, p_charge_max=0.8, p_discharge_max=0.8,
                  eta_charge=0.92, eta_discharge=0.92),   # medium battery
]
env_het = StorageArbitrageEnv(batteries=fleet)
obs, info = env_het.reset(seed=0)
print(f'Heterogeneous fleet: N={env_het.N} batteries')
print(f'  Obs shape   : {obs.shape}')
print(f'  Action shape: {env_het.action_space.shape}')
print(f'  Initial SoC : {info["soc_mwh"]} MWh')
env_het.close()

## 5. Run Baseline Policies

Three baselines bracket the performance range:
- **Random** — uniform action sampling (lower bound)  
- **Threshold** — rule-based buy-low/sell-high  
- **LP** — perfect foresight (theoretical upper bound, cheats with future prices)

In [ ]:
from random_policy import RandomPolicy, run_episodes
from threshold_policy import ThresholdPolicy, run_episodes as run_threshold_ep
from perfect_foresight import PerfectForesightLP, _get_episode_prices

N_EPISODES = 5
SEED       = 0

# --- Random Policy ---
env_r = StorageArbitrageEnv(n_batteries=1)
rp    = RandomPolicy(env_r, seed=SEED)
res_r = run_episodes(rp, env_r, n_episodes=N_EPISODES, seed=SEED)
print(f'Random    mean profit: ${res_r["mean_profit"]:>8.2f}  std: ${res_r["std_profit"]:.2f}')

# --- Threshold Policy ---
env_t = StorageArbitrageEnv(n_batteries=1)
tp    = ThresholdPolicy()
tp.reset(env=env_t)
res_t = run_threshold_ep(tp, env_t, n_episodes=N_EPISODES, seed=SEED)
print(f'Threshold mean profit: ${res_t["mean_profit"]:>8.2f}  std: ${res_t["std_profit"]:.2f}')

# --- LP Perfect Foresight ---
env_lp    = StorageArbitrageEnv(n_batteries=1)
lp_solver = PerfectForesightLP(env_lp)
lp_profits = []
for ep in range(N_EPISODES):
    obs_lp, info_lp = env_lp.reset(seed=SEED + ep)
    prices_lp = _get_episode_prices(env_lp)
    result_lp = lp_solver.solve(prices_lp, info_lp['soc_mwh'])
    lp_profits.append(result_lp['profit'])
lp_mean = float(np.mean(lp_profits))
print(f'LP        mean profit: ${lp_mean:>8.2f}  std: ${np.std(lp_profits):.2f}')

print()
print(f'Sanity check — random < threshold < LP:',
      res_r['mean_profit'] < res_t['mean_profit'] < lp_mean)
print(f'DRL target (70% LP): ${0.70 * lp_mean:.2f}')

## 6. Visualisation — Random Policy Episode

In [ ]:
from plots import plot_episode, plot_comparison

env_v = StorageArbitrageEnv(n_batteries=1)
res_r1 = run_episodes(RandomPolicy(env_v, seed=0), env_v, n_episodes=1, seed=42)
plot_episode(res_r1, title=f'Random Policy  |  profit=${res_r1["profits"][0]:.2f}')

## 7. Visualisation — Threshold Policy Episode

In [ ]:
env_v2 = StorageArbitrageEnv(n_batteries=1)
tp2 = ThresholdPolicy()
tp2.reset(env=env_v2)
res_t2 = run_episodes(tp2, env_v2, n_episodes=1, seed=42)
plot_episode(res_t2, title=f'Threshold Policy  |  profit=${res_t2["profits"][0]:.2f}')

## 8. Visualisation — LP Perfect Foresight Episode

In [ ]:
env_lp2  = StorageArbitrageEnv(n_batteries=1)
obs_lp2, info_lp2 = env_lp2.reset(seed=42)
prices_lp2 = _get_episode_prices(env_lp2)
lp_r2      = PerfectForesightLP(env_lp2).solve(prices_lp2, info_lp2['soc_mwh'])

T  = prices_lp2.shape[0]
dt = env_lp2.dt
lp_dict = {
    'profits':     [lp_r2['profit']],
    'all_soc':     [lp_r2['soc'][i]    for i in range(T+1)],
    'all_prices':  [prices_lp2[i]      for i in range(T)],
    'all_actions': [lp_r2['action'][i] for i in range(T)],
    'all_rewards': [-float(prices_lp2[i,0]) * float(lp_r2['action'][i,0]) * dt
                    for i in range(T)],
}
plot_episode(lp_dict,
             title=f'LP Perfect Foresight  |  profit=${lp_r2["profit"]:.2f}  (UPPER BOUND)')

## 9. Visualisation — Policy Comparison

All three policies on the same cumulative profit chart.

In [ ]:
# run one episode each with same seed for fair comparison
env_c  = StorageArbitrageEnv(n_batteries=1)

res_rc = run_episodes(RandomPolicy(env_c, seed=0), env_c, n_episodes=1, seed=42)
tp_c   = ThresholdPolicy(); tp_c.reset(env=env_c)
res_tc = run_episodes(tp_c, env_c, n_episodes=1, seed=42)

plot_comparison(
    {'Random': res_rc, 'Threshold': res_tc},
    save_path=None
)

## 10. MDP Summary

| Component | Definition |
|---|---|
| **State** | `s_t = [SoC_vector (N), price_vector (M), time_features (D)]` |
| **Action** | `a_t ∈ ℝᴺ` — charge/discharge power per battery |
| **Transition** | `R_{t+1,i} = clip(R_{t,i} + η_i·a_{t,i}·Δt, 0, R_max,i)` |
| **Reward** | `r_t = Σ_i[−p_{t,k(i)}·a_{t,i}·Δt − λ·|a_{t,i}|]` |
| **Episode** | `T = (24 × 60) ÷ 5 = 288 timesteps` |

The environment is registered as `StorageArbitrage-v0` and is fully Gymnasium-compatible.
Any SB3 algorithm (SAC, PPO, TD3) plugs in without modification.

In [ ]:
# Final summary
print('='*50)
print('Milestone 1 — Summary')
print('='*50)
print(f'Environment     : StorageArbitrageEnv (Gymnasium)')
print(f'Episode length  : T = (24×60)÷5 = 288 timesteps')
print(f'Baseline results (N=1, {N_EPISODES} episodes):')
print(f'  Random        : ${res_r["mean_profit"]:>8.2f}/day')
print(f'  Threshold     : ${res_t["mean_profit"]:>8.2f}/day')
print(f'  LP (ceiling)  : ${lp_mean:>8.2f}/day')
print(f'  70%% LP target : ${0.70*lp_mean:>8.2f}/day  (DRL acceptance criterion)')
print(f'Sanity check    : random < threshold < LP = {res_r["mean_profit"] < res_t["mean_profit"] < lp_mean}')